# Занятие 1, пара 2. Цвет, яркость и ваш проект

> **Первое занятие: сначала учебный результат.** Регистрация сегодня не нужна; GitHub нужен к следующему занятию. Git-шаги и их проверки сегодня выполняем только при работающем Git; иначе сохраняем обычные файлы. Основной маршрут не запускается из-за библиотек — откройте `00. Старт без установки.ipynb`, а без Jupyter — `../docs/СТАРТ_БЕЗ_УСТАНОВКИ.html`. Подробно: `../docs/ПЕРВЫЙ_СТАРТ.md`.
> Работайте в собственной копии, а не в общей папке курса. Выберите доступную для записи папку; сохраняйте отдельную копию результата.


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/smth_on_it

Сначала сохраните эту тетрадь к себе в проект и работайте в копии: «File → Save Notebook As…» и путь `cv_project/workbooks/01.2 Цвет, яркость и ваш проект.ipynb`.

### Подготовка

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

import cv2
import matplotlib.pyplot as plt
import numpy as np
from course_support.cvtools import imread, show
img = imread(DATA_DIR / 'hopper.jpg')

PROJECT_DIR


## Загадка синего лица

Помните портрет из первой пары?

In [ ]:
plt.imshow(img)
plt.axis('off');

OpenCV хранит цвета в порядке **B, G, R** — синий, зелёный, красный. А Matplotlib, как и почти все остальные программы, ждёт **R, G, B**. Синий и красный поменялись местами — вот небо и порыжело.

Почему OpenCV так странно устроен? Когда библиотеку создавали, порядок BGR был распространён у производителей камер и в Windows: например, формат BMP хранит пиксели именно так. Решение закрепилось, и менять его теперь поздно — сломается слишком много кода.

Исправить легко — переставить каналы:

In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # или просто img[:, :, ::-1]

plt.imshow(rgb)
plt.axis('off');

Путаница BGR и RGB — ошибка номер один в компьютерном зрении. Она подстерегает везде, где встречаются две библиотеки: OpenCV и Matplotlib, OpenCV и Pillow, OpenCV и нейросеть, которую обучали на RGB. Нейросеть, кстати, не упадёт с ошибкой — она просто станет тихо ошибаться чаще. Функция `show` переставляет каналы сама, поэтому дальше пользуемся ей.

Разложим портрет на каналы:

In [ ]:
blue, green, red = cv2.split(img)
show(blue, green, red, titles=['B — синий', 'G — зелёный', 'R — красный'])

Каждый канал — отдельная картинка в оттенках серого: чем светлее пиксель, тем больше в нём этого цвета. Небо светлое в синем канале, красные полосы флага — в красном, а лицо светлее всего в красном: кожа отражает больше красного света.

### Камера измеряет только треть

**Забавный факт.** В распространённых камерах с байеровским фильтром на каждом светочувствительном элементе стоит светофильтр одного цвета — красного, зелёного или синего. Значит, каждый пиксель измеряет только одно число из трёх, а два других камера досчитывает по соседям. Это та же интерполяция, и в телефоне она выполняется миллионы раз на каждом снимке. Такую схему фильтров придумал Брайс Байер из Kodak в 1970-х. Зелёных фильтров в ней вдвое больше, чем красных или синих: к зелёному глаз человека чувствительнее всего.

### Оттенки серого

Та же чувствительность глаза объясняет, как OpenCV переводит цвет в оттенки серого. Это не среднее трёх каналов, а взвешенная сумма:

$$Y = 0{,}299 \cdot R + 0{,}587 \cdot G + 0{,}114 \cdot B$$

Зелёный весит больше всех, синий — меньше всех. Проверим, что OpenCV считает именно так:

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
manual = 0.299 * red + 0.587 * green + 0.114 * blue

print('OpenCV:', gray[300, 256], '  вручную:', round(manual[300, 256], 1))
print('Наибольшее расхождение по всей картинке:', np.abs(gray - manual).max().round(2))

Расхождение меньше единицы: OpenCV считает в целых числах и округляет.

### Глаз обманывает, пиксель — нет

Посмотрите на картинку ниже. Полоска посередине светлее слева или справа?

In [ ]:
illusion = imread(COURSE_DIR / 'data' / 'illusion.png', cv2.IMREAD_GRAYSCALE)
show(illusion, axis=True, size=9)

Большинству людей кажется, что полоска светлее слева и темнее справа. Проверим не глазами, а числами.

### ✏️ Ваш ход 1

По осям графика определите, в каких строках и столбцах лежит полоска, и вырежьте её срезом в переменную `bar`: `illusion[строки, столбцы]`. Ячейка покажет наименьшее и наибольшее значение пикселей в срезе.

Если в срез попадёт фон, проверка это заметит — тогда сузьте границы.

In [ ]:
# ✏️ ваш код здесь
print('наименьшее:', bar.min(), ' наибольшее:', bar.max())

In [ ]:
check('01.2.1', bar)

Полоска везде одного цвета — 128, ровно середина шкалы. Обманул нас мозг: он оценивает яркость не саму по себе, а по сравнению с соседями. На тёмном фоне полоска кажется светлее, на светлом — темнее. Ещё более сильную иллюзию придумал Эдвард Адельсон — поищите «checker shadow illusion».

В жизни это полезно: белый лист мы узнаём и на солнце, и в тени, хотя пиксели у него совсем разные. Это называют константностью восприятия. Камера же честно измеряет пиксели и ничего не понимает. Научить её узнавать тот же лист в любом освещении — одна из задач компьютерного зрения.

**Забавный факт.** В феврале 2015 года интернет поссорился из-за фотографии платья: одни видели его сине-чёрным, другие — бело-золотым. Спорили даже учёные, а потом написали об этом научные статьи. Платье на самом деле сине-чёрное, а спор возник из-за того, какое освещение мозг «додумывает» за фотографией: тёплое или холодное. Камера телефона решает ту же задачу каждый раз, когда выставляет баланс белого.

### Осторожно: арифметика с uint8

Сделаем портрет ярче: прибавим к каждому пикселю 100.

In [ ]:
brighter_wrong = img + 100
show(img, brighter_wrong, titles=['оригинал', 'img + 100'])

Светлые места стали тёмными и цветными. Тип `uint8` вмещает числа только от 0 до 255, и при переполнении счёт начинается заново: 200 + 100 = 300, а в uint8 это 300 − 256 = 44. Так работают часы: 10 часов плюс 5 часов — это 3 часа, а не 15.

Правильно «упирать» значение в 255: посчитать в типе побольше и обрезать лишнее. Функции OpenCV вроде `cv2.add` делают это сами — такая арифметика называется арифметикой с насыщением.

In [ ]:
brighter = np.clip(img.astype(np.int16) + 100, 0, 255).astype(np.uint8)
show(img, brighter_wrong, brighter, titles=['оригинал', 'с переполнением', 'с насыщением'])

### Гистограмма: паспорт яркости

Гистограмма показывает, сколько на картинке пикселей каждой яркости. Это то самое распределение из теории вероятностей, только случайная величина здесь — яркость пикселя.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
axes[0].hist(gray.ravel(), bins=256, range=(0, 256), color='gray')
axes[0].set_title('оригинал')
axes[1].hist(cv2.cvtColor(brighter, cv2.COLOR_BGR2GRAY).ravel(), bins=256, range=(0, 256), color='gray')
axes[1].set_title('ярче на 100')
for ax in axes:
    ax.set_xlabel('яркость')
axes[0].set_ylabel('число пикселей');

Слева — тёмные пиксели мундира, справа — светлые. У осветлённого портрета гистограмма уехала вправо и упёрлась в край: огромный столбик у 255 — это пересвет, все эти пиксели стали одинаково белыми, и детали в них потеряны навсегда. В ручном режиме камеры телефона такую же гистограмму показывают прямо во время съёмки, чтобы вы заметили пересвет до того, как нажмёте кнопку.

### HSV: цвет, как его описывает человек

RGB удобен для экрана, но неудобен для вопросов вроде «найди всё красное»: у красного яблока в тени и на солнце совсем разные R, G и B. Поэтому часто переходят в пространство **HSV**:
- **H** (hue) — оттенок: какой это цвет;
- **S** (saturation) — насыщенность: насколько цвет чистый, а не серый;
- **V** (value) — яркость.

При простом снижении яркости V падает, а H может оставаться близким. Цвет освещения, блики и баланс белого меняют и H: HSV не делает метод универсальным.

**Забавный факт.** Оттенок обычно измеряют в градусах от 0 до 360 — это угол на цветовом круге. Но в `uint8` число 360 не помещается, поэтому OpenCV делит градусы пополам, и H у него идёт от 0 до 179. Красный — около 0 и около 179 (это же круг!), жёлтый — около 30, зелёный — около 60, синий — около 120.

Найдём на портрете всё синее. Функция `cv2.inRange` оставляет пиксели, у которых каждый канал попал в заданный диапазон, и возвращает **маску**: белое — подошло, чёрное — нет.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
blue_mask = cv2.inRange(hsv, (95, 80, 80), (125, 255, 255))
show(img, blue_mask, titles=['оригинал', 'маска «синее»'])

Небо и синее поле флага попали в маску, а тёмно-синий мундир — нет: у него слишком маленькая яркость V. Попробуйте поменять границы диапазона и посмотрите, что изменится.

### ✏️ Ваш ход 2: светофор

Вот три светофора:

In [ ]:
lights = [imread(COURSE_DIR / 'data' / 'traffic_lights' / f'{name}.png') for name in ['red', 'yellow', 'green']]
show(*lights, titles=['красный', 'жёлтый', 'зелёный'], size=2.5)

Допишите функцию `traffic_light(image)`: по картинке светофора она должна определять, какой сигнал горит, и возвращать строку `'красный'`, `'жёлтый'` или `'зелёный'`.

Проверка запустит вашу функцию на 12 новых светофорах — других размеров, на другом фоне и с шумом, — так что подогнать ответ под три картинки выше не получится.

Решений много. Две идеи на выбор:
1. **По положению.** Горящая лампа самая яркая. Разделите картинку по высоте на три части и посмотрите, в какой из них больше средняя яркость V. Пригодится `np.argmax`.
2. **По цвету.** Возьмите самые яркие и насыщенные пиксели и посмотрите на их оттенок H. Осторожно: красный живёт на обоих концах шкалы — около 0 и около 179.

In [ ]:
def traffic_light(image):
    """Возвращает 'красный', 'жёлтый' или 'зелёный' — какой сигнал горит."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    # ✏️ ваш код здесь


[traffic_light(light) for light in lights]

In [ ]:
check('01.2.2', traffic_light)

Кто решал по положению, а кто по цвету? Какой способ надёжнее? Подумайте, где сломается каждый: горизонтальный светофор (бывают и такие), светофор, снятый сбоку, солнце, которое бьёт прямо в лампы.

Вывод, который пригодится весь семестр: у задачи компьютерного зрения почти всегда несколько решений, и каждое работает в своих условиях. Поэтому решение проверяют не на трёх картинках, а на многих — ровно так, как только что сделала проверка.

## Ваш проект

### Какие задачи решает компьютерное зрение

| Задача | Что на выходе | Пример |
|---|---|---|
| Классификация | одна метка на всё изображение | «на фото — знак "Стоп"» |
| Детекция | рамки вокруг объектов и их метки | «на кадре три машины и пешеход» |
| Сегментация | метка для каждого пикселя | «вот пиксели дороги, а вот — тротуара» |
| Ключевые точки | координаты особых точек | углы документа, суставы человека |
| Трекинг | один и тот же объект на разных кадрах | «машина № 5 проехала перекрёсток» |
| Распознавание текста (OCR) | текст | номер дома, сумма в чеке |
| 3D | расстояние до точек, форма объекта | карта глубины, 3D-модель комнаты |

Классические задачи попробуем на занятиях. Семестровый проект — маленький честный эксперимент: **одна задача классификации, одни группы train/validation/test, сравнение признаков + ML с CNN**. Нейросеть не обязана победить: объяснение результата важнее красивого числа.

### Семестровый проект

Сегодня только предварительная идея. До следующего занятия можно подумать и выбрать своё; согласуем посильность и доступность данных. Лабораторные 4–6 — последовательные этапы, а не ещё один проект поверх них.

| Направление | Посильный классификационный вопрос |
|---|---|
| Дефекты | целая поверхность / дефект на фрагменте |
| Полка | место занято / пусто |
| Документ | ориентация или тип страницы |
| Городская среда | тип знака на вырезанном фрагменте |
| Видео | тип объекта на фрагменте кадра |
| Свой трек | доступные вам изображения и 2–6 различимых классов |

Подсчёт на видео, поиск товара и выравнивание документа — отдельные демонстрации/расширения, не дополнительные обязательные задачи каждому. Классы пока предположительные.

К следующему занятию принесите карточку идеи и до 10 пробных кадров, если тема уже ясна; если нет — два варианта с доступным источником. Не нужно снимать сотни похожих кадров: кадры одной серии нельзя разнести между обучением и тестом. Не публикуйте непроверенные личные фотографии. Подробнее — `../docs/ПРОЕКТ.md` и `../docs/ДАННЫЕ.md`.


### ✏️ Ваш ход 3

Запишите **предварительную** идею в словарь `my_project`:
- `'трек'` — любое понятное название направления (можно своё);
- `'вариант'` — что именно вы будете снимать, одной фразой;
- `'классы'` — список из 2–6 классов для будущей нейросети.

Хороший вариант конкретный: не «дефекты», а «трещины и сколы на керамической плитке».
Если пока не выбрали, возьмите пример для тренировки словаря, отметьте это в README и выберите тему дома. Зелёная проверка проверяет формат, а не утверждает тему.


In [ ]:
# ✏️ ваш код здесь
my_project

In [ ]:
check('01.2.3', my_project)

### ✏️ Ваш ход 4

Откройте `README.md` в корне **своего проекта**, запишите предварительный трек и вариант. Это рабочая гипотеза: до следующего занятия её можно поменять. Сохраните файл; коммит сегодня необязателен. Проверка видит сохранённый файл, но не оценивает качество темы.

In [ ]:
check('01.2.4', PROJECT_DIR, my_project)

## Сохраняем проект

Сначала Ctrl+S. Затем скопируйте **всю папку проекта** на свой носитель или доступное хранилище, откройте копию тетради и найдите последний ответ. Одного notebook недостаточно для автономности: ему нужны course_support и данные.

Храните работу в своей папке и имейте отдельную проверенную копию. Git — необязательное расширение первого блока; инструкция в `../docs/GIT.md`.

In [ ]:
print('Сохраните тетрадь (Ctrl+S), скопируйте папку проекта и откройте копию. Python не видит несохранённый текст в браузере.')

### GitHub — дома или на своём устройстве

К следующему занятию создайте свой репозиторий и загрузите полный проект, включая `course_support`. Публичность необязательна, преподавателю нужен доступ. Не оставляйте вход и токены на общем компьютере. Подробности — `../docs/GIT.md`. Количество коммитов не заменяет объяснение результата.

In [ ]:
print('GitHub подключим к следующему занятию. Сегодня проверяем сохранённые файлы.')

## Итоги занятия

- Цвета в OpenCV хранятся в порядке BGR, а почти все остальные программы ждут RGB.
- В байеровской матрице недостающие цветовые компоненты восстанавливаются. Восприятие зависит от контекста; числа пикселей тоже зависят от камеры и обработки.
- `uint8` переполняется, как часы. Для «найди всё красное» удобнее HSV.
- У вас есть предварительная идея проекта и проверенная копия работы.

In [ ]:
summary('01.2')

## Домашнее задание

К следующему занятию:

1. GitHub-аккаунт и отдельный репозиторий курса с README и сегодняшним результатом. Можно начать через сайт без установленного Git. Приватный репозиторий допустим, предоставьте преподавателю доступ.
2. Принесите карточку идеи: вопрос, доступные данные, ожидаемый результат и трудность; можно два варианта. Если тема уже ясна — до 10 пробных кадров разных серий; публиковать пока не нужно.
3. Прочитайте `../docs/ПЕРВЫЙ_СТАРТ.md`, критерии и описание проекта. Если регистрация не получается, сообщите конкретную проблему и принесите файлы с работой.

Сегодня регистрация не нужна. К следующей неделе она нужна всем; сдача файлом остаётся временным техническим резервом. Установка домашней среды и необязательное чтение не подменяют маленький выполненный результат.


## Полезное и интересное

- Игра: [Quick, Draw!](https://quickdraw.withgoogle.com/) — нейросеть пытается угадать ваш рисунок за 20 секунд. Посмотрите, что она «видит», а что нет.
- Иллюзия: [Checker shadow illusion](https://persci.mit.edu/gallery/checkershadow) Эдварда Адельсона — клетки A и B одного цвета. Проверьте пипеткой в любом редакторе.
- История: [The dress](https://en.wikipedia.org/wiki/The_dress) — статья о платье, которое поссорило интернет, и о том, что про него выяснили учёные.
- Соревнования: список — в `../docs/ПОЛЕЗНОЕ.md`. Участие добровольное, без баллов в ведомости.

Больше — в `../docs/ПОЛЕЗНОЕ.md`.